Test

This is a Test

In [1]:
import pandas as pd

In [2]:
# COLAB SETUP — safe to skip if running locally (generated by Claude). Allows different members of the project to run the code either in google colab or locally.
# This code snippet copies our project's github repo to the google colab environment.
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/overflow-prediction'):
        !git clone https://github.com/miirage-exe/overflow-prediction.git
    os.chdir('/content/overflow-prediction') # Change the current working directory so relative paths like "./data" etc... in the code below works.

Cloning into 'overflow-prediction'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 28 (delta 3), reused 22 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 18.94 MiB | 4.22 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (9/9), done.


## Data fetching and merging

In [3]:
# Fetch pluviometer and sewage data of a specific year from the files
def fetch_datasets(year):
  df_pluv = pd.read_excel(f'data/pluviometer/{year}.xlsx')
  df_sew = pd.read_csv(f'data/sewage/U24_{year}.csv')

  # --- Cleaning of pluviometer dataset

  # Renaming columns
  labels = df_pluv.columns[1:3] # Get second and third columns name (differs between 2022 and 2023)
  df_pluv = df_pluv.rename(columns={"FLOWBRU - PLUVIO ALL":"datetime", labels[0]:"precip_avant_port", labels[1]:"precip_flagey" })

  # Keep only usefull columns and rows
  df_pluv = df_pluv[["datetime", "precip_avant_port", "precip_flagey"]]
  df_pluv = df_pluv[1:-13] # Remove first row (labels in the excel file) and last one (totals per month)

  # Converting to datetime
  df_pluv['datetime'] = pd.to_datetime(df_pluv["datetime"], format='%d-%m-%Y %H:%M')

  # Sort the dataframe by datetime, to ensure chronological order
  df_pluv = df_pluv.sort_values(by="datetime").reset_index(drop=True) # We use .reset_index(drop=True), otherwise the old index will be added as a new column, which we do not want
  df_pluv = df_pluv.set_index("datetime") # Set datetime as index of the dataframe, meaning we will use it to reference the rows

  # --- Cleaning of sewage dataset

  # Renaming columns
  df_sew = df_sew.rename(columns={"Date":"datetime", "Value":"water_level" })

  # Converting to datetime
  df_sew['datetime'] = pd.to_datetime(df_sew["datetime"])

  # Sort the dataframe by datetime, to ensure chronological order
  df_sew = df_sew.sort_values(by="datetime").reset_index(drop=True) # We use .reset_index(drop=True), otherwise the old index will be added as a new column, which we do not want
  df_sew = df_sew.set_index("datetime") # Set datetime as index of the dataframe, meaning we will use it to reference the rows

  return df_pluv, df_sew

In [4]:
def merge_dataset(year, sample_rate):

  # TODO: Data cleaning.

  df_pluv, df_sew = fetch_datasets(year)
  df_pluv = df_pluv.resample(sample_rate).first() # Resample to hourly data, while keeping first measure of each hour
  df_sew = df_sew.resample(sample_rate).max() # Resample to hourly data, while keeping maximum level of each hour (worst case prediction)

  return df_pluv.merge(df_sew, on="datetime")

In [6]:
# For each year, merge pluviometer and sewage data
time_sample = '60min'

df_2022 = merge_dataset(2022, time_sample)
df_2023 = merge_dataset(2023, time_sample)
df_2024 = merge_dataset(2024, time_sample)

# Our final dataset, concatenation of each year.
df = pd.concat([df_2022, df_2023, df_2024])

,precip_avant_port,precip_flagey,water_level
datetime,,,
2024-12-30 19:00:00,0,0,1155.835
2024-12-30 20:00:00,0,0,1184.819
2024-12-30 21:00:00,0,0,1177.631
2024-12-30 22:00:00,0,0,1143.777
2024-12-30 23:00:00,0,0,1131.720


## Separation into training, validation, and testing datasets